In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [2]:
class Conv2D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, activation='linear'):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.weight = nn.Parameter(
            nn.init.kaiming_normal_(torch.empty(out_channels, in_channels, kernel_size, kernel_size))
        )
        self.bias = nn.Parameter(torch.zeros(out_channels))
        self.activation = activation
    def forward(self, x):
        batch_size, in_channels, height, width = x.size()
        out_height = (height + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_width = (width + 2 * self.padding - self.kernel_size) // self.stride + 1

        # Apply padding
        if self.padding > 0:
            x = F.pad(x, (self.padding, self.padding, self.padding, self.padding))

        # Unfold the input into patches (out_height * out_width = number of patches)
        x_unfolded = F.unfold(x, kernel_size=self.kernel_size, stride=self.stride)  # [batch_size, in_channels * kernel_size * kernel_size, out_height * out_width]

        # Perform convolution using matrix multiplication
        weight_flat = self.weight.view(self.out_channels, -1)                       # [out_channels, in_channels * kernel_size * kernel_size]
        output = weight_flat @ x_unfolded + self.bias.view(-1, 1)                   # [out_channels, out_height * out_width]

        # Reshape the output to the correct dimensions
        output = output.view(batch_size, self.out_channels, out_height, out_width)

        # Apply activation function
        if self.activation == 'relu':
            output = F.relu(output)
        elif self.activation == 'sigmoid':
            output = F.sigmoid(output)

        return output

In [3]:
class MaxPooling2D(nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding

    def forward(self, x):
        batch_size, in_channels, height, width = x.size()
        out_height = (height + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_width = (width + 2 * self.padding - self.kernel_size) // self.stride + 1

        # Apply padding
        if self.padding > 0:
            x = F.pad(x, (self.padding, self.padding, self.padding, self.padding))

        # Unfold the input into patches
        x_unfolded = F.unfold(x, kernel_size=self.kernel_size, stride=self.stride)  # [batch_size, in_channels * kernel_size * kernel_size, out_height * out_width]

        # Reshape to [batch_size, in_channels, kernel_size * kernel_size, out_height * out_width]
        x_unfolded = x_unfolded.view(batch_size, in_channels, self.kernel_size * self.kernel_size, -1)

        # Perform max pooling
        output = x_unfolded.max(dim=2)[0]  # [batch_size, in_channels, out_height * out_width]

        # Reshape the output to the correct dimensions
        output = output.view(batch_size, in_channels, out_height, out_width)

        return output

In [4]:
class AveragePooling2D(nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding

    def forward(self, x):
        batch_size, in_channels, height, width = x.size()
        out_height = (height + 2 * self.padding - self.kernel_size) // self.stride + 1
        out_width = (width + 2 * self.padding - self.kernel_size) // self.stride + 1

        # Apply padding
        if self.padding > 0:
            x = F.pad(x, (self.padding, self.padding, self.padding, self.padding))

        # Unfold the input into patches
        x_unfolded = F.unfold(x, kernel_size=self.kernel_size, stride=self.stride)  # [batch_size, in_channels * kernel_size * kernel_size, out_height * out_width]

        # Reshape to [batch_size, in_channels, kernel_size * kernel_size, out_height * out_width]
        x_unfolded = x_unfolded.view(batch_size, in_channels, self.kernel_size * self.kernel_size, -1)

        # Perform average pooling
        output = x_unfolded.mean(dim=2)  # [batch_size, in_channels, out_height * out_width]

        # Reshape the output to the correct dimensions
        output = output.view(batch_size, in_channels, out_height, out_width)

        return output

In [5]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels, out_channels, num_classes=None):
        super().__init__()
        self.branch1 = Conv2D(in_channels, out_channels // 4, kernel_size=1, activation='relu')        # 1x1 convolution
        self.branch2 = nn.Sequential(
            Conv2D(in_channels, out_channels // 4, kernel_size=1, activation='relu'),                  # 1x1 convolution
            Conv2D(out_channels // 4, out_channels // 4, kernel_size=3, padding=1, activation='relu')  # 3x3 convolution with padding
        )
        self.branch3 = nn.Sequential(
            Conv2D(in_channels, out_channels // 4, kernel_size=1, activation='relu'),                  # 1x1 convolution
            Conv2D(out_channels // 4, out_channels // 4, kernel_size=5, padding=2, activation='relu')  # 5x5 convolution with padding
        )
        self.branch4 = nn.Sequential(
            MaxPooling2D(kernel_size=3, stride=1, padding=1),                                          # 3x3 max pooling with stride 1
            Conv2D(in_channels, out_channels // 4, kernel_size=1, activation='relu')                   # 1x1 convolution after pooling
        )
        if num_classes is not None:
            self.branch5 = nn.Sequential(
                AveragePooling2D(kernel_size=5, stride=3, padding=0),                                  # 5x5 average pooling with stride 3
                Conv2D(in_channels, out_channels // 4, kernel_size=1, activation='relu'),              # 1x1 convolution after pooling
                nn.AdaptiveMaxPool2d((1, 1)),                                                          # Adaptive max pooling to get a fixed-size output
                nn.Flatten(),                                                                          # Flatten the output
                nn.Linear(out_channels // 4, out_channels // 4),                                       # Fully connected layer
                nn.Linear(out_channels // 4, num_classes),                                             # Fully connected layer for classification
            )

    def forward(self, x):
        branch1_out = self.branch1(x)
        branch2_out = self.branch2(x)
        branch3_out = self.branch3(x)
        branch4_out = self.branch4(x)

        output = torch.cat([branch1_out, branch2_out, branch3_out, branch4_out], dim=1)

        if hasattr(self, 'branch5'):
            branch5_out = self.branch5(x)
            return output, branch5_out

        return output

In [6]:
class InceptionNetwork(nn.Module):
    def __init__(self, num_classes=10, input_channels=3):
        super().__init__()
        self.conv1 = Conv2D(in_channels=input_channels, out_channels=64, kernel_size=7, stride=2, padding=3, activation='relu')
        self.pool1 = MaxPooling2D(kernel_size=3, stride=2, padding=1)
        self.norm1 = nn.BatchNorm2d(64)
        self.conv2 = Conv2D(in_channels=64, out_channels=64, kernel_size=1, activation='relu')
        self.conv3 = Conv2D(in_channels=64, out_channels=192, kernel_size=3, padding=1, activation='relu')
        self.norm2 = nn.BatchNorm2d(192)
        self.inception1 = InceptionModule(in_channels=192, out_channels=256)
        self.inception2 = InceptionModule(in_channels=256, out_channels=512)
        self.inception3 = InceptionModule(in_channels=512, out_channels=512)
        self.inception4 = InceptionModule(in_channels=512, out_channels=512, num_classes=num_classes)
        self.inception5 = InceptionModule(in_channels=512, out_channels=512)
        self.inception6 = InceptionModule(in_channels=512, out_channels=512)
        self.inception7 = InceptionModule(in_channels=512, out_channels=512, num_classes=num_classes)
        self.inception8 = InceptionModule(in_channels=512, out_channels=512)
        self.inception9 = InceptionModule(in_channels=512, out_channels=512)
        self.pool2 = AveragePooling2D(kernel_size=7, stride=1)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.norm1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.norm2(x)
        x = self.inception1(x)
        x = self.inception2(x)
        x = self.inception3(x)
        x, aux1 = self.inception4(x)  # Auxiliary classifier 1
        x = self.inception5(x)
        x = self.inception6(x)
        x, aux2 = self.inception7(x)  # Auxiliary classifier 2
        x = self.inception8(x)
        x = self.inception9(x)
        x = self.pool2(x)
        x = self.flatten(x)
        output = self.fc(x)

        return output, aux1, aux2

    def backward(self, output, aux1, aux2, target):
        # Compute the loss for the main output and auxiliary classifiers
        main_loss = F.cross_entropy(output, target)
        aux1_loss = F.cross_entropy(aux1, target)
        aux2_loss = F.cross_entropy(aux2, target)

        # Combine the losses (you can adjust the weights as needed)
        total_loss = main_loss + 0.3 * aux1_loss + 0.3 * aux2_loss

        # Backpropagate the total loss
        total_loss.backward()

        return total_loss.item()

    def fit(self, X, y, epochs=10, lr=0.001, print_every=1):
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)
        for epoch in range(epochs):
            self.train()
            optimizer.zero_grad()
            output, aux1, aux2 = self.forward(X)
            total_loss = self.backward(output, aux1, aux2, y)
            optimizer.step()
            if epoch % print_every == 0:
                print(f"Epoch [{epoch}/{epochs}], Loss: {total_loss:.4f}")

In [7]:

import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

indices = np.random.choice(len(X_train), 128, replace=False)
X_batch_keras = X_train[indices]
y_batch_keras = y_train[indices]

X_batch = torch.tensor(X_batch_keras, dtype=torch.float32) / 255.0
X_batch = X_batch.unsqueeze(1)  # [128, 28, 28] -> [128, 1, 28, 28]

y_batch = torch.tensor(y_batch_keras, dtype=torch.long)

print(f"X shape: {X_batch.shape}")  # [128, 1, 28, 28]
print(f"y shape: {y_batch.shape}")  # [128,]

model = InceptionNetwork(num_classes=10, input_channels=1)

# Train
print("\nPrediction: ")
model.fit(X_batch, y_batch, epochs=5, lr=0.001, print_every=1)

X shape: torch.Size([128, 1, 28, 28])
y shape: torch.Size([128])

Prediction: 
Epoch [0/5], Loss: 4.0260
Epoch [1/5], Loss: 10.7125
Epoch [2/5], Loss: 4.4641
Epoch [3/5], Loss: 3.7419
Epoch [4/5], Loss: 3.6505
